# Day 2 — LangGraph State, Tools, Streaming Progress and Human-in-the-Loop

We build two complementary patterns:
1. a **controlled workflow** where deterministic nodes decide when tools run; and
2. a **native tool-calling loop** where the model chooses tools.

This distinction matters in real systems. LangGraph is especially useful when deterministic workflow steps, model-driven reasoning, persistence and human review must coexist.

>**Dr Julius Sechang Mboli**
>
>**DAIM, University of Hull**
>
>**https://www.hull.ac.uk/staff-directory/julius-mboli**
>
>**https://www.linkedin.com/in/engr-julius-sechang-mboli/**
>
>**https://jsmboli.github.io/**

In [ ]:
from pathlib import Path
import os, sys, json, time, importlib.util
import pandas as pd
pd.set_option('display.max_colwidth', None)

# Locate the package without relying on a fixed working directory.
cwd = Path.cwd().resolve()
RESOURCE_DIR = None
for p in [cwd, *cwd.parents]:
    if (p / "resources" / "src").exists():
        RESOURCE_DIR = p / "resources"
        break
    if (p / "src").exists() and (p / "notebooks").exists() and (p / "data").exists():
        RESOURCE_DIR = p
        break
if RESOURCE_DIR is None:
    raise RuntimeError("Could not locate resources/src. Extract the complete bootcamp ZIP and open this notebook from inside it.")

SRC = RESOURCE_DIR / "src"
DATA = RESOURCE_DIR / "data"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from providers import ProviderRouter, ProviderError, make_messages
from notebook_utils import (
    environment_table, provider_table, response_table,
    run_with_progress, progress_indicator, display_response, display_note,
)

router = ProviderRouter(verbose=True)
display(environment_table(RESOURCE_DIR, router))

In [2]:
from tools import calculator, pick_rate, knowledge_lookup, risk_assessment, retrieve_observations
from langgraph_agents import build_workflow_agent, build_native_tool_agent

In [ ]:
status = run_with_progress("Checking Groq, Ollama and optional providers", router.diagnose, True)
display(provider_table(status))

PRIMARY_PROVIDER = (
    "groq" if status["groq"]["available"]
    else "ollama" if status["ollama"]["available"]
    else "mimic"
)
print("Primary live provider for this notebook:", PRIMARY_PROVIDER)
print("Groq model:", status["groq"].get("model"))
print("Ollama model:", status["ollama"].get("model"))

## 1. Inspect deterministic tools before giving them to an agent

Never hide basic business logic inside a prompt if a deterministic function can implement it transparently. We test each tool independently first.

Top_k=3 is a text generation setting that restricts an LLM to choosing only from the top 3 most likely next words.

In [4]:
print("calculator ->", calculator("735 / 6.5"))
print("pick_rate ->", pick_rate(735, 6.5))
print("low-risk request ->", risk_assessment("Summarise common packing exception categories for training."))
print("high-risk request ->", risk_assessment("Use a named employee's pick-rate performance to draft a disciplinary warning."))
display(pd.DataFrame(knowledge_lookup("human review risky agent tool calls", top_k=3)))

calculator -> 113.07692307692308
pick_rate -> {'items_picked': 735, 'hours': 6.5, 'pick_rate_per_hour': 113.08, 'interpretation_warning': 'Do not use this metric alone for judging a person. Add safety, quality, task mix, training, fatigue, equipment, and human review.'}
low-risk request -> {'risk': 'medium', 'signals': 'none', 'recommendation': 'continue_with_logging'}
high-risk request -> {'risk': 'high', 'signals': 'performance, disciplinary, employee, named', 'recommendation': 'human_review'}


,topic,source_type,content
0,evaluation,testing_note,"Evaluate agents using representative tasks, ed..."
1,agentic_ai,teaching_note,An AI agent is a software system that uses an ...
2,human_in_loop,responsible_ai,Human-in-the-loop is used when an action may b...


## 2. Controlled LangGraph workflow

The graph executes:

`START → assess risk → retrieve evidence → calculate → LLM draft → human-review gate → finalise → END`

Each node has one clear responsibility. The graph prints node-level progress, and the model call reports its provider/model. This makes a long-running cell explain **what it is doing** rather than appearing frozen.

In [5]:
if PRIMARY_PROVIDER == "mimic":
    print("No live provider available; use Day 0 to configure Groq or Ollama for this graph.")
else:
    app = build_workflow_agent(provider=PRIMARY_PROVIDER, progress=True)
    config = {"configurable":{"thread_id":"day2-controlled-low-risk"}}
    with progress_indicator("Running controlled LangGraph workflow"):
        low = app.invoke({
            "user_request":"Use the training observations to propose a bounded packing-exception assistant."
        }, config=config)
    print("\nFinal answer:\n", low.get("final"))
    print("\nLLM metadata:")
    display(pd.DataFrame([low.get("llm_metadata",{})]))
    print("\nPersisted state keys:", sorted(app.get_state(config).values.keys()))

[1/6] Assessing request risk…
[2/6] Retrieving grounded teaching evidence…
[3/6] Running deterministic calculations where required…
[4/6] Synthesising a grounded draft with provider=groq…
▶ Groq request started | model=qwen/qwen3.6-27b | max_tokens=700
✓ Groq completed in 1.623s | model=qwen/qwen3.6-27b
[5/6] Applying the human-review policy gate…
[6/6] Finalising output and preserving review decision…

Final answer:
 Based on the provided training observations and responsible AI guidelines, here is the proposal for a **Bounded Packing-Exception Assistant**.

### 1. Design Rationale (Retrieved Guidance)
*   **Agent vs. Workflow:** Per `workflow_vs_agent`, packing processes are largely predictable. Therefore, the assistant should operate within a **workflow boundary** (fixed steps for standard packing) but use **agent flexibility** only when exceptions (damage, weight mismatches) occur, as the next step depends on ambiguous observations.
*   **Human-in-the-Loop (HITL):** Per `human_in_l

,provider,model,latency_s,input_tokens,output_tokens,total_tokens,finish_reason,request_id,endpoint,reasoning_mode
0,groq,qwen/qwen3.6-27b,1.623,756,700,1456,length,chatcmpl-c00591ff-2678-4ed9-bf9e-1528a06a96b4,https://api.groq.com/openai/v1/chat/completions,format=hidden; effort=none



Persisted state keys: ['draft', 'evidence', 'final', 'llm_metadata', 'metric', 'review_decision', 'risk', 'user_request']


## 3. Genuine Human-in-the-Loop pause and resume

A disclaimer at the end of an answer is not HITL. Here `interrupt()` actually suspends execution and the checkpointer preserves state. The same `thread_id` is then resumed using `Command(resume=...)`.

The review packet includes the proposed draft, risk, provider and model so the reviewer knows what generated the recommendation.

In [6]:
if PRIMARY_PROVIDER != "mimic":
    from langgraph.types import Command
    hitl_app = build_workflow_agent(provider=PRIMARY_PROVIDER, progress=True)
    hitl_config={"configurable":{"thread_id":"day2-hitl"}}
    with progress_indicator("Running until the human-review interrupt"):
        first = hitl_app.invoke({
            "user_request":"Use a named employee's pick-rate performance to decide whether the employee should receive a disciplinary warning."
        }, config=hitl_config)
    interrupts=first.get("__interrupt__", [])
    print("Interrupt count:", len(interrupts))
    for item in interrupts:
        print(item.value if hasattr(item,"value") else item)
else:
    print("Configure Groq or Ollama to run the real interrupt example.")

[1/6] Assessing request risk…
[2/6] Retrieving grounded teaching evidence…
[3/6] Running deterministic calculations where required…
[4/6] Synthesising a grounded draft with provider=groq…
▶ Groq request started | model=qwen/qwen3.6-27b | max_tokens=700
✓ Groq completed in 0.749s | model=qwen/qwen3.6-27b
[5/6] Applying the human-review policy gate…
Interrupt count: 1
{'reason': 'High-risk request requires human review before release or action.', 'request': "Use a named employee's pick-rate performance to decide whether the employee should receive a disciplinary warning.", 'risk': {'risk': 'high', 'signals': 'performance, disciplinary, employee, named', 'recommendation': 'human_review'}, 'draft': '**Observations**\nThe request involves using specific performance data (pick-rate) for a named employee to determine a disciplinary outcome. The risk assessment flags this as **high risk** due to the sensitive nature of employment actions and the potential for irreversible harm.\n\n**Retrieved 

In [7]:
# Simulated authorised reviewer decision for teaching.
if PRIMARY_PROVIDER != "mimic" and first.get("__interrupt__"):
    reviewer_decision={
        "decision":"edit",
        "edited_text":"Do not make a disciplinary decision from pick rate. Review quality, safety, task mix, training, equipment and contextual evidence with an authorised human decision-maker."
    }
    with progress_indicator("Resuming graph after human review"):
        reviewed=hitl_app.invoke(Command(resume=reviewer_decision), config=hitl_config)
    print(reviewed["final"])

[5/6] Applying the human-review policy gate…
[6/6] Finalising output and preserving review decision…
Do not make a disciplinary decision from pick rate. Review quality, safety, task mix, training, equipment and contextual evidence with an authorised human decision-maker.


## 4. Native tool calling with Groq

Now the model, rather than the workflow, decides whether to call `lookup_guidance`, `lookup_visit_observations`, `calculate_pick_rate` or `assess_request_risk`. The tool loop is closer to a ReAct-style agent and is useful for comparing flexibility against control.

Groq is recommended for this exercise because the hosted model has stronger tool-use capability and much lower latency than the tiny local models installed for the financially inclusive route.

In [8]:
if not status["groq"]["available"]:
    display_note(status["groq"].get("detail","Groq unavailable"), "warning")
else:
    from langchain_core.messages import HumanMessage
    native = build_native_tool_agent(provider="groq", progress=True)
    with progress_indicator("Groq LangGraph native tool-calling loop"):
        native_out = native.invoke({"messages":[HumanMessage(content=(
            "Retrieve relevant training guidance, calculate the pick rate for 735 items over 6.5 hours, "
            "assess the risk of using that number to judge a named employee, and answer with the appropriate human-review boundary."
        ))]})
    rows=[]
    for i,m in enumerate(native_out["messages"]):
        rows.append({
            "index":i,"message_type":type(m).__name__,"content":getattr(m,"content","")[:350],
            "tool_calls":getattr(m,"tool_calls",None)
        })
    display(pd.DataFrame(rows))

[model] Invoking groq model for tool selection/synthesis…
[tool] lookup_guidance(query='training guidance pick rate performance metrics employee evaluation')
[model] Invoking groq model for tool selection/synthesis…
[tool] calculate_pick_rate(items=735.0, hours=6.5)
[model] Invoking groq model for tool selection/synthesis…
[tool] assess_request_risk(…)
[model] Invoking groq model for tool selection/synthesis…


,index,message_type,content,tool_calls
0,0,HumanMessage,"Retrieve relevant training guidance, calculate...",None
1,1,AIMessage,,"[{'name': 'lookup_guidance', 'args': {'query':..."
2,2,ToolMessage,"[{'topic': 'agentic_ai', 'source_type': 'teach...",None
3,3,AIMessage,,"[{'name': 'calculate_pick_rate', 'args': {'hou..."
4,4,ToolMessage,"{'items_picked': 735.0, 'hours': 6.5, 'pick_ra...",None
5,5,AIMessage,,"[{'name': 'assess_request_risk', 'args': {'req..."
6,6,ToolMessage,"{'risk': 'high', 'signals': 'employee, named',...",None
7,7,AIMessage,,[]


## 5. Optional local tool-calling experiment

Ollama itself supports tool calling, but **model capability matters**. A 0.8B or 1B model may chat successfully while making weak or malformed tool choices. That is a capability result, not an installation failure.

The previous pack also had a code defect here: a locally imported `MessagesState` appeared only in a postponed type annotation, causing `NameError`. The graph factory has been corrected.

In [9]:
if not status["ollama"]["available"]:
    print("Ollama unavailable from this kernel.")
else:
    try:
        from langchain_core.messages import HumanMessage
        local_native=build_native_tool_agent(provider="ollama", progress=True)
        with progress_indicator(f"Ollama native tool calling — {status['ollama']['model']}"):
            local_out=local_native.invoke({"messages":[HumanMessage(content="Calculate 500 items over 5 hours using the correct tool and state the interpretation warning.")]})
        for m in local_out["messages"]:
            print(type(m).__name__, "->", getattr(m,"content","")[:600], "tools=", getattr(m,"tool_calls",None))
    except Exception as exc:
        print("Local native-tool experiment did not complete reliably:", type(exc).__name__, exc)
        print("Use the controlled workflow pattern for small local models; try a larger tool-capable Ollama model for the extension.")

[model] Invoking ollama model for tool selection/synthesis…
[tool] calculate_pick_rate(items=500.0, hours=5.0)
[model] Invoking ollama model for tool selection/synthesis…
HumanMessage -> Calculate 500 items over 5 hours using the correct tool and state the interpretation warning. tools= None
AIMessage ->  tools= [{'name': 'calculate_pick_rate', 'args': {'items_picked': 500, 'hours': 5}, 'id': 'bd650850-1b37-410d-9f2b-a6c17f0b47d6', 'type': 'tool_call'}]
ToolMessage -> {'items_picked': 500.0, 'hours': 5.0, 'pick_rate_per_hour': 100.0, 'interpretation_warning': 'Do not use this metric alone for judging a person. Add safety, quality, task mix, training, fatigue, equipment, and human review.'} tools= None
AIMessage -> Based on the calculation:

**Pick Rate:** 500 items over 5 hours = **100.0 items per hour**

The interpretation warning states that this metric should not be used in isolation when making employment decisions, as it doesn't account for safety, quality of work, task mix, train

## 6. Engineering reflection

For each node, ask:
- Why is this an LLM decision rather than deterministic code?
- What state must be persisted?
- What can fail and how is it retried?
- What should be logged for auditability?
- Where must a person approve, edit or reject?
- Would the same graph behave acceptably after a provider outage or model upgrade?